# 02: Iterative Adversarial Attacks (PGD, Carlini-Wagner & DeepFool)
**Comparative Study of Multi-Step First-Order & Optimization-Based Adversaries**

---

## 1. Attack Formulations

### 1.1 Projected Gradient Descent (PGD) — Madry et al. (2018)
Considered the gold-standard first-order adversary. It runs multi-step gradient ascent with random uniform initialization inside the $L_\infty$ $\epsilon$-ball:
$$\mathbf{x}^0 = \mathbf{x} + \mathcal{U}(-\epsilon, \epsilon)$$
$$\mathbf{x}^{t+1} = \Pi_{\mathcal{B}_\epsilon(\mathbf{x})}\left( \mathbf{x}^t + lpha \cdot \mathrm{sign}\left(
abla_{\mathbf{x}^t} \mathcal{L}(oldsymbol{	heta}, \mathbf{x}^t, y)ight) ight)$$

### 1.2 Carlini-Wagner $L_2$ Attack — Carlini & Wagner (2017)
Formulates attack as a continuous constrained optimization problem using a change of variables $\mathbf{x}' = rac{1}{2}(	anh(\mathbf{w}) + 1)$:
$$\min_{\mathbf{w}} \|\mathbf{x}' - \mathbf{x}\|_2^2 + c \cdot \max\left( \max_{i 
eq y} Z(\mathbf{x}')_i - Z(\mathbf{x}')_y, -\kappa ight)$$

### 1.3 DeepFool — Moosavi-Dezfooli et al. (2016)
Iteratively projects towards the closest decision boundary by linearizing classifier hyperplanes.


In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from adv_studio.models import get_model
from adv_studio.data import get_mnist_loaders, get_sample_digits
from adv_studio.attacks import FGSMAttack, IFGSMAttack, PGDAttack, CarliniWagnerL2Attack, DeepFoolAttack
from adv_studio.evaluation import compute_robust_accuracy, compute_distortion_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model("simple_cnn", pretrained_path="../checkpoints/mnist_cnn.pth", device=device)
model.eval()

_, test_loader = get_mnist_loaders(data_dir="../DATA", batch_size=64)


## 2. Multi-Attack Benchmark Comparison


In [ ]:
attacks = {
    "FGSM (eps=0.2)": FGSMAttack(model, epsilon=0.2, device=device),
    "I-FGSM (eps=0.2, T=10)": IFGSMAttack(model, epsilon=0.2, steps=10, device=device),
    "PGD-Linf (eps=0.2, T=20)": PGDAttack(model, epsilon=0.2, steps=20, random_start=True, device=device),
    "DeepFool": DeepFoolAttack(model, max_iter=30, device=device),
    "Carlini-Wagner L2": CarliniWagnerL2Attack(model, steps=30, lr=0.02, device=device),
}

print(f"{'Attack':<26} | {'Robust Acc':<12} | {'ASR':<10} | {'Mean Conf':<12}")
print("-" * 65)

for name, atk in attacks.items():
    res = compute_robust_accuracy(model, test_loader, attack=atk, device=device, max_batches=10)
    print(f"{name:<26} | {res['robust_accuracy']*100:>10.2f}% | {res['attack_success_rate']*100:>8.2f}% | {res['mean_adv_confidence']*100:>10.2f}%")
